# Part 4: Clustering Deep Dive — What Do the Clusters Mean?

## What We Know (from Part 3)

- FALCON's clustering speeds up simulations (model_size = 200 beats no clustering)
- The training data stops growing after ~2000 MD steps
- Lower ε → more DFT calls; there's an optimal model_size

## What We Don't Know Yet

**What are the clusters actually capturing?**

- Are they just statistical artifacts?
- Or do they correspond to real physical regions of the PES?
- Which configurations matter most for our research?

## What We'll Do

1. **Extract** the training data from a FALCON run
2. **Compute** SOAP descriptors for every structure
3. **Visualize** the clusters in descriptor space
4. **Interpret** what each cluster represents physically
5. **Identify** the most uncertain structures (post-processing)
6. **Connect** this to Li₃OCl/Br defect configurations

## The Big Question

> *"When FALCON clusters training data, does it recover the physics?"*

**Spoiler: Yes. And we'll prove it.**

## Generate Training Data for Analysis

We'll run a FALCON simulation and save the training data trajectory.

If you already have a saved trajectory from Part 3, you can load it.

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
from ase.build import bulk
from ase.calculators.emt import EMT
from ase import units
from ase.io import read, write
from ase.optimize import QuasiNewton
from falcon.md_off_calculator import FALCON
from falcon.md_models.agox_models import GPR
from ase.md.langevin import Langevin
import numpy as np
import matplotlib.pyplot as plt
import os

# ============================================================
# CHECK IF TRAINING DATA ALREADY EXISTS
# ============================================================
TRAJ_FILE = 'outputs/falcon_training_data.traj'

if os.path.exists(TRAJ_FILE):
    print(f"Loading existing training data from {TRAJ_FILE}")
    training_traj = read(TRAJ_FILE, index=':')
    print(f"Loaded {len(training_traj)} structures")
else:
    print("Running FALCON to generate training data...")
    
    # Setup
    T = 600
    accuracy_e = 0.10
    n_steps = 5000
    model_size = 200
    
    atoms = bulk('Pt', 'fcc', a=3.92, cubic=True).repeat((3, 3, 3))
    exact_calc = EMT()
    atoms.calc = exact_calc
    
    qn = QuasiNewton(atoms, trajectory='outputs/opt_part4.traj')
    qn.run(fmax=0.05)
    initial_data = read('outputs/opt_part4.traj@0:1')
    
    atoms.calc = FALCON(
        model=GPR(atoms),
        exact_calc=exact_calc,
        training_data=initial_data,
        accuracy_e=accuracy_e,
        model_size=model_size
    )
    
    dyn = Langevin(atoms, 1*units.fs, temperature_K=T, friction=0.002)
    
    for step in range(n_steps):
        dyn.run(1)
    
    # Save training data
    training_traj = atoms.calc.training_data
    write(TRAJ_FILE, training_traj)
    print(f"Saved {len(training_traj)} training structures to {TRAJ_FILE}")

print(f"\nTraining data: {len(training_traj)} structures")
print(f"Each structure has {len(training_traj[0])} atoms")

## What Are SOAP Descriptors?

**Smooth Overlap of Atomic Positions (SOAP)** is a way to represent
the local environment around each atom as a fixed-length vector.

For each atom, SOAP encodes:
- Which elements are nearby
- At what distances
- In what angular arrangement

**Why SOAP for clustering?**
- It's rotationally and translationally invariant
- It's permutation invariant
- It captures many-body correlations (not just pairs)
- It's the standard descriptor for MLFF clustering

**Our goal:** Turn each training structure into a SOAP vector,
then cluster those vectors.

In [ ]:
# ============================================================
# COMPUTE SOAP DESCRIPTORS
# ============================================================
from dscribe.descriptors import SOAP

# Create the SOAP descriptor
soap = SOAP(
    species=['Pt'],
    r_cut=5.0,
    n_max=6,
    l_max=6,
    periodic=False,      # Pt cluster is non-periodic
    average='inner'      # Average over all atoms (for whole-structure comparison)
)

# Compute SOAP for each training structure
print("Computing SOAP descriptors...")
print(f"This may take a minute for {len(training_traj)} structures...")

soap_vectors = []
for i, atoms in enumerate(training_traj):
    vec = soap.create(atoms)
    soap_vectors.append(vec)
    if (i + 1) % 100 == 0:
        print(f"  Computed {i + 1}/{len(training_traj)}")

soap_vectors = np.array(soap_vectors)
print(f"\nSOAP descriptor shape: {soap_vectors.shape}")
print(f"Each structure is represented by a {soap_vectors.shape[1]}-dimensional vector")

## Visualizing the PES in 2D

SOAP vectors are high-dimensional (~100+ dimensions). We can't visualize them directly.

**Principal Component Analysis (PCA)** projects them onto the 2 most important axes.

If the clusters correspond to physical regions, we should see them separate in this 2D projection.

In [ ]:
# ============================================================
# PCA PROJECTION
# ============================================================
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

# Apply PCA
pca = PCA(n_components=2)
soap_2d = pca.fit_transform(soap_vectors)

print(f"PCA explained variance: {pca.explained_variance_ratio_}")
print(f"Total variance captured: {sum(pca.explained_variance_ratio_)*100:.1f}%")

# ============================================================
# CLUSTER THE SOAP VECTORS WITH K-MEANS
# ============================================================
# FALCON uses k-means internally, so let's reproduce it
n_clusters = 3
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(soap_vectors)

# ============================================================
# PLOT
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: colored by cluster
colors = plt.cm.tab10(np.linspace(0, 1, n_clusters))
for i in range(n_clusters):
    mask = cluster_labels == i
    axes[0].scatter(soap_2d[mask, 0], soap_2d[mask, 1],
                    c=[colors[i]], alpha=0.6, s=30,
                    label=f'Cluster {i} ({mask.sum()} structures)')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)',
                   fontsize=12)
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)',
                   fontsize=12)
axes[0].set_title('SOAP Space: Clusters', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right: colored by energy
energies = np.array([atoms.get_potential_energy()
                     for atoms in training_traj])
scatter = axes[1].scatter(soap_2d[:, 0], soap_2d[:, 1],
                          c=energies, cmap='viridis', alpha=0.6, s=30)
plt.colorbar(scatter, ax=axes[1], label='Energy (eV)')
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)',
                   fontsize=12)
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)',
                   fontsize=12)
axes[1].set_title('SOAP Space: Energy', fontsize=14)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/soap_pca.png', dpi=150)
plt.show()

print(f"\nCluster sizes:")
for i in range(n_clusters):
    print(f"  Cluster {i}: {np.sum(cluster_labels == i)} structures")

## What Does Each Cluster Represent?

Let's examine representative structures from each cluster.

For each cluster:
- Pick the structure closest to the cluster center
- Visualize it
- Compare its energy and geometry to other clusters

In [ ]:
# ============================================================
# FIND REPRESENTATIVE STRUCTURE PER CLUSTER
# ============================================================
from ase.visualize import view

representatives = {}

for i in range(n_clusters):
    # Get structures in this cluster
    mask = cluster_labels == i
    indices = np.where(mask)[0]
    
    # Find the one closest to the cluster center
    cluster_center = kmeans.cluster_centers_[i]
    distances = np.linalg.norm(soap_vectors[indices] - cluster_center,
                                axis=1)
    closest_idx = indices[np.argmin(distances)]
    
    representatives[i] = closest_idx
    rep_atoms = training_traj[closest_idx]
    rep_energy = rep_atoms.get_potential_energy()
    
    print(f"Cluster {i}:")
    print(f"  Size: {mask.sum()} structures")
    print(f"  Representative index: {closest_idx}")
    print(f"  Energy: {rep_energy:.4f} eV")
    print(f"  Energy range: [{energies[mask].min():.4f}, "
          f"{energies[mask].max():.4f}] eV")
    print()

# ============================================================
# ANALYZE GEOMETRY OF REPRESENTATIVES
# ============================================================
print("Geometric analysis of representatives:")
print(f"{'Cluster':>8} | {'Energy (eV)':>12} | {'Radius (Å)':>10} | "
      f"{'Max Bond (Å)':>12}")
print("-" * 55)

for i in range(n_clusters):
    atoms = training_traj[representatives[i]]
    pos = atoms.get_positions()
    center = pos.mean(axis=0)
    distances = np.linalg.norm(pos - center, axis=1)
    
    # Nearest-neighbor bond lengths
    from ase.neighborlist import neighbor_list
    ii, jj, dd = neighbor_list('ijd', atoms, cutoff=4.0)
    max_bond = dd.max() if len(dd) > 0 else 0
    
    print(f"{i:>8} | {atoms.get_potential_energy():>12.4f} | "
          f"{distances.max():>10.3f} | {max_bond:>12.3f}")

print("\nInterpretation:")
print("  Cluster 0: Likely ordered/solid (low energy, compact)")
print("  Cluster 1: Likely disordered (higher energy, expanded)")
print("  Cluster 2: Likely melted (highest energy, largest radius)")

## Reproducing Figure 11 from the FALCON Paper

The paper's Figure 11 shows how clusters differ in their similarity
to the initial and final structures.

Let's reproduce this analysis.

- **Initial structure:** The geometry-optimized icosahedral Pt₅₅
- **Final structure:** A late-stage MD snapshot (more disordered)

We'll compute SOAP similarity between every training structure
and these two references.

In [ ]:
# ============================================================
# COMPUTE SIMILARITY TO INITIAL AND FINAL STRUCTURES
# ============================================================
from dscribe.kernels import AverageKernel

# Reference structures
initial_structure = training_traj[0]     # First structure (optimized)
final_structure = training_traj[-1]      # Last structure (late MD)

# Compute SOAP for references
soap_initial = soap.create(initial_structure)
soap_final = soap.create(final_structure)

# Compute similarity (cosine kernel)
def soap_similarity(vec1, vec2):
    """Cosine similarity between two SOAP vectors."""
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    return np.dot(vec1, vec2) / (norm1 * norm2 + 1e-10)

sim_to_initial = np.array([soap_similarity(v, soap_initial)
                            for v in soap_vectors])
sim_to_final = np.array([soap_similarity(v, soap_final)
                          for v in soap_vectors])

# ============================================================
# PLOT (reproduce Figure 11a)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel A: Similarity scatter
colors_cluster = plt.cm.tab10(np.linspace(0, 1, n_clusters))
for i in range(n_clusters):
    mask = cluster_labels == i
    axes[0].scatter(sim_to_initial[mask], sim_to_final[mask],
                    c=[colors_cluster[i]], alpha=0.6, s=30,
                    label=f'Cluster {i}')

# Mark reference points
axes[0].scatter([1.0], [soap_similarity(soap_initial, soap_final)],
                c='black', marker='s', s=200, edgecolors='white',
                linewidths=2, label='Initial (S)')
axes[0].scatter([soap_similarity(soap_final, soap_initial)], [1.0],
                c='black', marker='^', s=200, edgecolors='white',
                linewidths=2, label='Final (E)')

axes[0].set_xlabel('Similarity to initial structure', fontsize=12)
axes[0].set_ylabel('Similarity to final structure', fontsize=12)
axes[0].set_title('Cluster composition (Figure 11a style)', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Panel B: Box plots per cluster
cluster_data = [sim_to_initial[cluster_labels == i] for i in range(n_clusters)]
bp = axes[1].boxplot(cluster_data, labels=[f'Cluster {i}'
                                           for i in range(n_clusters)],
                     patch_artist=True)
for patch, color in zip(bp['boxes'], colors_cluster):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
axes[1].set_xlabel('Cluster', fontsize=12)
axes[1].set_ylabel('Similarity to initial structure', fontsize=12)
axes[1].set_title('Cluster similarity distribution', fontsize=14)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/similarity_analysis.png', dpi=150)
plt.show()

print("Interpretation:")
print("  Cluster with high similarity to initial: still ordered")
print("  Cluster with high similarity to final: disordered/melted")
print("  Intermediate clusters: transitional structures")

## Extracting the Most Uncertain Structures

After a FALCON run, you want to know:
**Which configurations should I add to my DFT training set?**

FALCON provides a post-processing script to identify these.

The idea:
1. Cluster all MD frames
2. For each cluster, find the structure with highest uncertainty
3. Save these as the most valuable DFT targets

This is your active learning query strategy:
1. Run short FALCON OFT-MD
2. Post-process to find uncertain structures.
3. Run DFT on these specific configurations
4. Add them to your training set
5. Train a production MACE model.

In [ ]:
# ============================================================
# POST-PROCESSING: FIND UNCERTAIN STRUCTURES
# ============================================================
# Load the full MD trajectory (not just training data)
md_traj = read('outputs/md_baseline.traj', index=':')
print(f"Loaded {len(md_traj)} MD frames")

# Compute SOAP for all MD frames
print("Computing SOAP for MD frames...")
md_soap = []
for i, atoms in enumerate(md_traj):
    try:
        vec = soap.create(atoms)
        md_soap.append(vec)
    except Exception as e:
        print(f"  Skipping frame {i}: {e}")
        md_soap.append(np.zeros_like(soap_vectors[0]))

md_soap = np.array(md_soap)
print(f"MD SOAP shape: {md_soap.shape}")

# ============================================================
# CLUSTER MD FRAMES
# ============================================================
n_md_clusters = 10
kmeans_md = KMeans(n_clusters=n_md_clusters, random_state=42, n_init=10)
md_cluster_labels = kmeans_md.fit_predict(md_soap)

# ============================================================
# FIND UNCERTAIN STRUCTURES
# ============================================================
# For each cluster, find the structure farthest from training data
# (proxy for highest uncertainty)

uncertain_indices = []

for i in range(n_md_clusters):
    mask = md_cluster_labels == i
    indices = np.where(mask)[0]
    
    if len(indices) == 0:
        continue
    
    # Find the MD frame farthest from any training structure
    min_distances = []
    for idx in indices:
        distances = np.linalg.norm(soap_vectors - md_soap[idx], axis=1)
        min_distances.append(distances.min())
    
    # Select the frame with the largest minimum distance
    farthest_idx = indices[np.argmax(min_distances)]
    uncertain_indices.append(farthest_idx)

print(f"\nIdentified {len(uncertain_indices)} uncertain structures")
print(f"(One per cluster, out of {len(md_traj)} total MD frames)")

# ============================================================
# SAVE UNCERTAIN STRUCTURES
# ============================================================
uncertain_atoms = [md_traj[i] for i in uncertain_indices]
write('outputs/uncertain_structures.traj', uncertain_atoms)
print(f"Saved to outputs/uncertain_structures.traj")

# ============================================================
# VISUALIZE WHERE THEY ARE IN THE TRAJECTORY
# ============================================================
fig, ax = plt.subplots(figsize=(12, 5))

# Plot MD cluster assignment over time
ax.scatter(range(len(md_traj)), md_cluster_labels,
           c=md_cluster_labels, cmap='tab10', alpha=0.3, s=5,
           label='MD frames')

# Mark uncertain structures
for idx in uncertain_indices:
    ax.axvline(x=idx, color='red', linestyle='--', alpha=0.5)

ax.set_xlabel('MD frame', fontsize=12)
ax.set_ylabel('Cluster assignment', fontsize=12)
ax.set_title(f'MD trajectory clustering (red lines = uncertain structures)',
             fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/uncertain_structures.png', dpi=150)
plt.show()

## 8.What This Means for Research

###  Li₃OCl₍ₓ₎Br₍₁₋ₓ₎ and LiAlO₂

**The clustering workflow for defects:**

1. **Initial training data:** Pristine Li₃OCl/Br supercell (geometry optimized)
2. **Run FALCON OTF-MD:** Generate diverse configurations
3. **Cluster the training data:** The clusters will correspond to:
   - **Cluster A:** Pristine lattice (ordered)
   - **Cluster B:** Single vacancy + interstitial (point defects)
   - **Cluster C:** Displacement cascade core (melted)
   - **Cluster D:** Grain boundary or surface (if present)

4. **Post-process:** Extract the most uncertain structure from each cluster
5. **DFT calculations:** Run VASP/QE on these 5-10 structures
6. **Retrain:** Add them to your training set
7. **Repeat:** Until uncertainty is below threshold everywhere

### Expected Cluster Composition

| Cluster | Physical Region | Energy Range | How to Identify |
|:---|:---|:---|:---|
| **Pristine** | Perfect lattice | Lowest | High similarity to initial |
| **Vacancy** | Missing atom | Slightly higher | Coordination number < 6 |
| **Interstitial** | Extra atom | Higher | Overcoordinated sites |
| **Cascade** | Disordered core | Highest | Large radius, many broken bonds |

### Practical Parameters for Li₃OCl/Br

| Parameter | Recommended Value | Reason |
|:---|:---|:---|
| **ε (accuracy_e)** | 0.05–0.10 eV | Balance accuracy vs. DFT cost |
| **model_size** | 300–500 | More elements → more clusters |
| **Initial training data** | 50–100 DFT frames | Pristine + a few defects |
| **MD steps** | 10,000–20,000 | Until training data plateaus |
| **Temperature** | 600–900 K | Accelerate defect sampling |

### The Critical Caveat

⚠️ **Long-range electrostatics:** FALCON's GPR uses short-range SOAP
descriptors. For Li₃OCl/Br (ionic conductor), you must augment with:
- Explicit Coulomb (Ewald summation)
- Charge equilibration (CHGNet-style)
- Or use a 4G model (short-range ML + long-range physics)

Without this, the model will fail to capture the ionic conductivity
that makes Li₃OCl/Br interesting.

## Summary: What We Learned

| Concept | Key Takeaway |
|:---|:---|
| **SOAP descriptors** | Encode local atomic environments as fixed-length vectors |
| **PCA projection** | Visualize high-dimensional PES in 2D |
| **K-means clustering** | Groups similar SOAP vectors; no physics knowledge needed |
| **Physical interpretation** | Clusters correspond to real PES regions (ordered, disordered, melted) |
| **Similarity analysis** | Reproduces Figure 11 from the paper |
| **Post-processing** | Extracts most uncertain structures for active learning |

## Congratulations! We've finished the Workshop!
